In [ ]:
# ==============================================================================
# 1. SETUP DO EXPERIMENTO E CALIBRAÇÃO DE COMPARAÇÕES (τc)
# ==============================================================================

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Hiperparâmetros metodológicos
N = 500_000   # Tamanho da amostra interna (número de operações por repetição)
R = 40        # Número de repetições globais

# Setup de variáveis e constantes em memória
v_int1 = 42
v_int2 = 42
v_int3 = 99
v_flt1 = 3.14159
v_flt2 = 3.14159
v_flt3 = 2.71828
c_int = 42
c_flt = 3.14159

# Funções isoladas para cada primitiva de comparação da tabela 4.2
def op_cte_int_eq_cte_int():
    for _ in range(N):
        1 == 1

def op_var_int_eq_cte_int():
    for _ in range(N):
        v_int1 == 42

def op_var_int_eq_var_int():
    for _ in range(N):
        v_int1 == v_int2

def op_var_flt_eq_var_flt():
    for _ in range(N):
        v_flt1 == v_flt2

def op_var_int_ne_var_int():
    for _ in range(N):
        v_int1 != v_int3

def op_var_int_gt_var_int():
    for _ in range(N):
        v_int3 > v_int1

def op_var_flt_lt_cte_flt():
    for _ in range(N):
        v_flt3 < 3.14159

# Mapeamento das operações
comparacoes = {
    "Igualdade: cte int == cte int": op_cte_int_eq_cte_int,
    "Igualdade: var int == cte int": op_var_int_eq_cte_int,
    "Igualdade: var int == var int": op_var_int_eq_var_int,
    "Igualdade: var float == var float": op_var_flt_eq_var_flt,
    "Desigualdade: var int != var int": op_var_int_ne_var_int,
    "Relacional: var int > var int": op_var_int_gt_var_int,
    "Relacional: var float < cte float": op_var_flt_lt_cte_flt
}

# Coleta e cálculo estatístico
tempos_brutos = {}
tempos_filtrados = {}
cortes_percentil = {}
tabela_resultados = []

print("Executando calibração de comparações (τc)...")

for nome, func in comparacoes.items():
    times_r = []
    for r in range(R):
        t0 = time.perf_counter()
        func()
        t1 = time.perf_counter()
        # Tempo unitário da primitiva por operação em segundos
        times_r.append((t1 - t0) / N)
    
    raw = np.array(times_r)
    tempos_brutos[nome] = raw
    
    # 1. Métricas Brutas
    mu_b = np.mean(raw)
    sigma_b = np.std(raw, ddof=1)
    cv_b = sigma_b / mu_b
    
    # 2. Filtragem por Percentil (cortando 5% e 95%)
    p5 = np.percentile(raw, 5)
    p95 = np.percentile(raw, 95)
    cortes_percentil[nome] = (p5, p95)
    
    fil = raw[(raw >= p5) & (raw <= p95)]
    tempos_filtrados[nome] = fil
    
    # 3. Métricas Filtradas
    mu_f = np.mean(fil)
    sigma_f = np.std(fil, ddof=1)
    cv_f = sigma_f / mu_f
    
    tabela_resultados.append({
        "Operação e Tipagem Exata": nome,
        "µbruta (s)": f"{mu_b:.4e}",
        "σbruta (s)": f"{sigma_b:.4e}",
        "CVbruto": f"{cv_b:.4f}",
        "µfil(τc) (s)": f"{mu_f:.4e}",
        "σfil (s)": f"{sigma_f:.4e}",
        "CVfil": f"{cv_f:.4f}",
        "< 0.15": "Sim" if cv_f < 0.15 else "Não"
    })

# Exibição da Matriz de Calibração homologada
df_tau_c = pd.DataFrame(tabela_resultados)
print("\n--- 4.2. MATRIZ DE CALIBRAÇÃO (τc) ---")
display(df_tau_c)


In [ ]:
# ==============================================================================
# 2. PAINEL GRÁFICO DE ESTABILIDADE (SEÇÃO 3 DO RELATÓRIO)
# ==============================================================================

cores_op = {nome: cor for nome, cor in zip(comparacoes.keys(), plt.cm.tab10(np.linspace(0, 1, len(comparacoes))))}
reps = np.arange(1, R + 1)

# ------------------------------------------------------------------------------
# 1) Gráfico de Dispersão Temporizada: r vs Tempo Acumulado / Unitário
# Evidencia a variabilidade temporal e picos anômalos (SO / Garbage Collector)
# ------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 6))

for nome in comparacoes.keys():
    ax.scatter(reps, tempos_brutos[nome], color=cores_op[nome], label=nome, alpha=0.85, s=35)
    ax.plot(reps, tempos_brutos[nome], color=cores_op[nome], alpha=0.35, lw=1.2)

ax.set_xlabel("Iteração / Repetição (r)")
ax.set_ylabel("Tempo Unitário τc (s)")
ax.set_title("1. Dispersão Temporizada dos Tempos de Comparação (τc)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, title="Primitiva de Comparação")
fig.subplots_adjust(right=0.68)
plt.show()

# ------------------------------------------------------------------------------
# 2) Histograma de Dados Brutos
# Mostra assimetria e caudas longas causadas por ruído sistêmico
# ------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 6))

for nome in comparacoes.keys():
    ax.hist(tempos_brutos[nome], bins=12, alpha=0.45, color=cores_op[nome], label=nome, edgecolor="white")

ax.set_xlabel("Tempo Unitário τc (s)")
ax.set_ylabel("Frequência")
ax.set_title("2. Histograma de Dados Brutos (Assimetria e Caudas Longas)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, title="Primitiva de Comparação")
fig.subplots_adjust(right=0.68)
plt.show()

# ------------------------------------------------------------------------------
# 3) Histograma Pós-Filtragem (Percentil 5% e 95%)
# Mostra a distribuição após filtro com linhas de corte visíveis
# ------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 6))

for nome in comparacoes.keys():
    ax.hist(tempos_filtrados[nome], bins=10, alpha=0.45, color=cores_op[nome], label=f"{nome} (Filtrado)", edgecolor="white")

# Linha de corte de exemplo para evidenciar visualmente o corte do percentil
op_exemplo = "Igualdade: var int == var int"
p5, p95 = cortes_percentil[op_exemplo]
ax.axvline(p5, color="red", ls="--", lw=1.2, label=f"Corte P5% ({p5:.2e}s)")
ax.axvline(p95, color="darkred", ls="--", lw=1.2, label=f"Corte P95% ({p95:.2e}s)")

ax.set_xlabel("Tempo Unitário Filtrado τc (s)")
ax.set_ylabel("Frequência")
ax.set_title("3. Histograma Pós-Filtragem (Percentis 5% - 95% com Linhas de Corte)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, title="Primitivas e Linhas de Corte")
fig.subplots_adjust(right=0.68)
plt.show()


In [ ]:
# ==============================================================================
# 3. PAINEL INTEGRADO 1x3 PARA ANEXAR NO RELATÓRIO (EXIGÊNCIA SEÇÃO 3)
# ==============================================================================
# Gera o painel de 3 gráficos para uma primitiva representativa (ou para todas)

def plot_painel_primitiva(nome_primitiva):
    raw = tempos_brutos[nome_primitiva]
    fil = tempos_filtrados[nome_primitiva]
    p5, p95 = cortes_percentil[nome_primitiva]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Painel Gráfico de Estabilidade — {nome_primitiva}", fontsize=14, fontweight="bold")
    
    # 1. Dispersão Temporizada
    axes[0].scatter(reps, raw, color="#3B6BB5", s=40, edgecolor="white", alpha=0.9)
    axes[0].plot(reps, raw, color="#3B6BB5", alpha=0.4, lw=1)
    axes[0].set_xlabel("Iteração (r)")
    axes[0].set_ylabel("Tempo τc (s)")
    axes[0].set_title("1. Dispersão Temporizada (Picos SO/GC)")
    axes[0].grid(True, linestyle=":", alpha=0.6)
    
    # 2. Histograma Bruto
    axes[1].hist(raw, bins=12, color="#6B6B6B", edgecolor="white", alpha=0.8)
    axes[1].set_xlabel("Tempo Bruto (s)")
    axes[1].set_ylabel("Frequência")
    axes[1].set_title("2. Histograma Bruto (Assimetria / Cauda Longa)")
    axes[1].grid(True, linestyle=":", alpha=0.6)
    
    # 3. Histograma Pós-Filtragem
    axes[2].hist(fil, bins=10, color="#C1553B", edgecolor="white", alpha=0.8, label="Filtrado (5%-95%)")
    axes[2].axvline(p5, color="black", ls="--", lw=1.2, label=f"P5% ({p5:.2e}s)")
    axes[2].axvline(p95, color="black", ls=":", lw=1.2, label=f"P95% ({p95:.2e}s)")
    axes[2].set_xlabel("Tempo Filtrado (s)")
    axes[2].set_ylabel("Frequência")
    axes[2].set_title("3. Histograma Pós-Filtragem (Normalidade)")
    axes[2].legend(frameon=False)
    axes[2].grid(True, linestyle=":", alpha=0.6)
    
    plt.tight_layout()
    plt.show()

# Exemplo: Painel de 3 gráficos para Igualdade de variáveis inteiras
plot_painel_primitiva("Igualdade: var int == var int")
